In [1]:
#%%
import gurobi as gp
import openpyxl
import pandas as pd
import plotly.express as px

ModuleNotFoundError: No module named 'gurobi'

In [4]:
import os 
import numpy as np

ModuleNotFoundError: No module named 'plotly'

In [ ]:

#%% Import Data
wb = openpyxl.load_workbook(os.path.realpath(os.path.join(__file__,"..","..","Data","data_updated.xlsx")))

# Sets
ws_sets = wb['Sets']
W = [i+1 for i in range(ws_sets['A2'].value)]
T = [i+1 for i in range(ws_sets['B2'].value)]
F = [i+1 for i in range(ws_sets['C2'].value)]

# Wind Farm Data
ws_wf = wb['WindFarm']
# w_size = {w: ws_wf[f"B{w+1}"].value for w in W }
# w_age = {w: ws_wf[f"C{w+1}"].value for w in W }
wx = {w: ws_wf[f"N{w+1}"].value for w in W }
wy = {w: ws_wf[f"O{w+1}"].value for w in W }
wd = {w: ws_wf[f"J{w+1}"].value for w in W }
ww = {w: ws_wf[f"L{w+1}"].value for w in W }
wt = {w: ws_wf[f"K{w+1}"].value for w in W }

# Facility Data
ws_f = wb['FacilityData']
co = {t: ws_f[f"C{t+1}"].value for t in T}
ca = {t: ws_f[f"B{t+1}"].value for t in T}

# Delivery Cost
dc = 47.5

#%% Model
model = gp.Model('model')
model.Params.NonConvex = 2
model.Params.MIPGap = 0.07


# Decision Vars
fb = model.addVars(F,T, vtype=gp.GRB.BINARY, name='var_build_facility_FT')
fx = model.addVars(F,T, ub=90, lb=-90, name='var_facility_lat_FT')
fy = model.addVars(F,T, ub=180, lb=-180, name='var_facility_lon_FT')

dist_x = model.addVars(W,F,T, name='var_facility_dist_lat_WFT')
dist_y = model.addVars(W,F,T, name='var_facility_dist_lon_WFT')
dist = model.addVars(W,F,T, name='var_facility_dist_WFT')
dw = model.addVars(W,F,T, name='var_deliver_waste_WFTY')


# Demand
c_d = model.addConstrs((gp.quicksum(dw[(w,f,t)] for t in T for f in F)
                        == wd[w] for w in W),
                        name = 'con_demand')

# Capacity
c_c = model.addConstrs((gp.quicksum(dw[(w,f,t)] * ww[w] for w in W)
                        <= ca[t] * fb[(f,t)] for f in F for t in T),
                        name = 'con_capacity')

# Distance Constraints
c_distx1 = model.addConstrs((dist_x[(w,f,t)] 
                             >= wx[w] - fx[(f,t)] 
                             for w in W for f in F for t in T),
                             name='con_dist_x_1')
c_distx2 = model.addConstrs((dist_x[(w,f,t)] 
                             >= -(wx[w] - fx[(f,t)]) 
                             for w in W for f in F for t in T),
                             name='con_dist_x_2')
c_disty1 = model.addConstrs((dist_y[(w,f,t)] 
                             >= wy[w] - fy[(f,t)] 
                             for w in W for f in F for t in T),
                             name='con_dist_y_1')
c_disty2 = model.addConstrs((dist_y[(w,f,t)] 
                             >= -(wy[w] - fy[(f,t)]) 
                             for w in W for f in F for t in T),
                             name='con_dist_y_2')
c_dist = model.addConstrs((dist[(w,f,t)] 
                           == dist_x[(w,f,t)]
                           + dist_y[(w,f,t)]
                           for w in W for f in F for t in T),
                           name='con_dist')

# Objective Function
cost_del = gp.quicksum(dist[(w,f,t)] * dw[(w,f,t)] * dc * 50
                       for w in W for f in F for t in T)
cost_build = gp.quicksum(co[t] * fb[(f,t)]
                         for f in F for t in T)
model.setObjective( cost_del + cost_build, sense=gp.GRB.MINIMIZE )

model.optimize()
for var in model.getVars():
    if var.X >= 0.01:
        print(f"{var.varName}: {var.X}")

#%% 
print(f"Total cost of delivery over 50 years: {cost_del.getValue()/1000000:.2f}")
print(f"Cost of building facilities: {cost_build.getValue()/1000000:.2f}")

# %% 

loc_data = pd.DataFrame([wx.values()]+[wy.values()]).T.rename(columns={0:"X",1:"Y"})
# loc_data = pd.DataFrame([wx.values()]+[wy.values()]+[wt.values()]).T.rename(columns={0:"X",1:"Y",2:'Yearly_Waste'})
loc_data['type'] = 'wind_farm'


for f in F:
    for t in T:
        if fb[(f,t)].X >= 0.01:
            loc_data = pd.concat([loc_data,
                                  pd.DataFrame([fx[(f,t)].X, fy[(f,t)].X, f"Facility {t}"],index=['X',"Y",'type']).T])

fig = px.scatter(loc_data, x="X",y="Y",color='type')
fig.show(renderer='browser')                             
# %%
